The goal of this notebook is to perform time series EDA. This is intended as EDA ONLY. (Not testing other classes, not performing data analysis etc.)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit

from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error as rmse

#now more time-series specific things
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.api as sm

import pmdarima as pm

system_ids = [2105, 2107, 7333, 9068] #the relevant ids
#system_id = 2105 #may have to look through them eventually
#path = f'../../../../data_ds_project/systems/prize/{system_id}/'

def systemPath(system_id):
    return f'../../../../data_ds_project/systems/prize/{system_id}/'

In [2]:
#Instead of using all the data/files for EDA, we will use only some
#we begin with a list of inverters, and then add on the meters, to make sure we're choosing from all of the possibilities
individual_data = pd.read_csv('../../../../data_ds_project/systems/prize/new_inverter_names_for_prize_cleaned_power.csv')
individual_data

,system_id,old_name,new_name
0,2107,inv_01_ac_power_inv_149583,0
1,2107,inv_02_ac_power_inv_149588,1
2,2107,inv_03_ac_power_inv_149593,2
3,2107,inv_04_ac_power_inv_149598,3
4,2107,inv_05_ac_power_inv_149603,4
...,...,...,...
141,9068,inverter_module_2.2_ac_power_(kw)_inv_150140,5
142,9068,inverter_module_2.3_ac_power_(kw)_inv_150141,6
143,9068,inverter_module_2.4_ac_power_(kw)_inv_150142,7
144,9068,inverter_1_ac_power_(kw)_inv_150143,8


In [3]:
meters_df = pd.DataFrame({
    'system_id' : [2105, 9068], 
    'old_name' : ['meter','meter'], 
    'new_name' : ['000','000']
})
all_datas = pd.concat([individual_data, meters_df], ignore_index = True)
all_datas

,system_id,old_name,new_name
0,2107,inv_01_ac_power_inv_149583,0
1,2107,inv_02_ac_power_inv_149588,1
2,2107,inv_03_ac_power_inv_149593,2
3,2107,inv_04_ac_power_inv_149598,3
4,2107,inv_05_ac_power_inv_149603,4
...,...,...,...
143,9068,inverter_module_2.4_ac_power_(kw)_inv_150142,7
144,9068,inverter_1_ac_power_(kw)_inv_150143,8
145,9068,inverter_2_ac_power_(kw)_inv_150144,9
146,2105,meter,000


In [4]:
#sample 10 of them. Set a random seed too to make this reproducible 
sample_data = all_datas.sample(10, random_state = 10)
sample_data

,system_id,old_name,new_name
79,7333,sos-03-109-inv1-power-kw__176646,55
124,7333,sos-06-081-inv1-power-kw__176691,100
24,7333,sos-01-001-inv1-power-kw__176591,0
35,7333,sos-01-044-inv1-power-kw__176602,11
85,7333,sos-04-040-inv1-power-kw__176652,61
59,7333,sos-02-026-inv1-power-kw__176626,35
10,2107,inv_11_ac_power_inv_149633,10
114,7333,sos-05-101-inv1-power-kw__176681,90
140,9068,inverter_module_2.1_ac_power_(kw)_inv_150139,4
144,9068,inverter_1_ac_power_(kw)_inv_150143,8


## Naive model 

Build a naive seasonal model. 

- train test split
- next cycle = average of all previous cycles. (do this by day)

In [18]:
from extract_and_clean import Clean

#run it on all of them! -- but with proportions: amount generated/capacity.
# estimate capacity by using maximum daily average value
all_errors = np.zeros(len(sample_data))
for i in range(len(sample_data)):

    #import file from 0th row

    #file name
    file_name = str(sample_data.iloc[i,0])+"_"
    if sample_data.iloc[i,1]=="meter":
        file_name = file_name + "meter_"
    else:
        file_name = file_name + "inverter_"
    file_name = file_name + str(sample_data.iloc[i,2]).zfill(3) + ".csv"

    df = pd.read_csv(f'../../../../data_ds_project/systems/prize/prize_cleaned_power/{file_name}')
    print(f"System_id = {str(sample_data.iloc[i,0])}")

    #turn strings into daytime
    df['time'] = pd.to_datetime(df['time'])

    #should check whether there are any skipped days
    obj = Clean()
    missing_days = obj.missing_days(df)
    if len(missing_days)!=0:
        print("Missing days: (there is a gap before printed day)")
        display(missing_days)

    #make a reasonable estimate as to the capacity -- I don't think we actually need to do this if we're not trying to extrapolate to new locations
    max = df['power'].max()
    df['proportion'] = df['power']/max

    #train test split
    X_train = df.iloc[:int(0.8*len(df))]
    X_test = df.iloc[int(0.8*len(df)):]

    #k-cross validation, since you gotta
    n_splits = 3
    kfold = TimeSeriesSplit(n_splits = n_splits)

    errors = np.zeros(n_splits) #errors array
    for j,(ind_train, ind_ho) in enumerate(kfold.split(X_train)):
        X_tt = X_train.iloc[ind_train]
        X_tt['power'] = X_tt['power']/max
        X_ho = X_train.iloc[ind_ho]
        y_ho = X_train.iloc[ind_ho,2]/max

        grouped_data_means = pd.DataFrame(X_tt.groupby(X_tt["time"].dt.strftime("%m-%d"))["power"].mean())
        #display(grouped_data_means)
        grouped_data_means = grouped_data_means.squeeze()

        X_ho["month_day"] = X_ho["time"].dt.strftime("%m-%d")
        X_ho["y_pred"] = X_ho["month_day"].map(grouped_data_means)
        y_pred = X_ho['y_pred']
        display(y_pred)
        y_pred[y_pred.isna()]=0 #some of the days are missing -- likely due to low reader output

        errors[j] = rmse(y_pred, y_ho)

    errors
    all_errors[i] = errors.mean()
all_errors

System_id = 7333
Missing days: (there is a gap before printed day)


,time,power,date_diff
814,2019-02-08,986.223438,16 days
876,2019-04-17,1551.919883,7 days
952,2019-07-03,1505.328784,2 days
1085,2019-12-19,810.772723,37 days
1087,2020-01-01,145.592523,12 days
1663,2021-08-11,1523.026393,13 days
1729,2021-10-26,1061.640267,11 days
1804,2022-01-14,590.345011,6 days
2026,2022-08-25,1594.252859,2 days


494    0.468118
495    0.428264
496    0.292510
497    0.671993
498    0.774856
         ...   
980    0.786038
981    0.732857
982    0.586854
983    0.287656
984    0.469238
Name: y_pred, Length: 491, dtype: float64

985     0.719015
986     0.742444
987     0.715733
988     0.736365
989     0.727284
          ...   
1471    0.390260
1472    0.383761
1473    0.467273
1474    0.452992
1475    0.241220
Name: y_pred, Length: 491, dtype: float64

1476    0.454334
1477    0.505155
1478    0.457043
1479    0.553247
1480    0.565803
          ...   
1962    0.858238
1963    0.871175
1964    0.848068
1965    0.789533
1966    0.822693
Name: y_pred, Length: 491, dtype: float64

...

,time,power,date_diff
2,2017-08-28,1362.256410,3 days
12,2017-09-15,140.086957,9 days
69,2017-11-13,972.534884,3 days
138,2018-01-24,856.000000,4 days
295,2018-07-02,1215.803922,3 days
1282,2021-03-23,1130.113636,9 days
1317,2021-04-28,1215.985075,2 days
1364,2021-06-16,842.052632,3 days
1368,2021-06-21,1403.489796,2 days
1408,2021-08-01,1289.000000,2 days


535     0.826088
536     0.627502
537     0.782580
538     0.840891
539     0.833943
          ...   
1063    0.739416
1064    0.876925
1065    0.869431
1066    0.881791
1067    0.852087
Name: y_pred, Length: 533, dtype: float64

1068    0.875675
1069    0.754275
1070    0.811019
1071    0.610348
1072    0.557782
          ...   
1596    0.628625
1597    0.251634
1598    0.297792
1599    0.459355
1600    0.271348
Name: y_pred, Length: 533, dtype: float64

1601    0.334581
1602    0.436756
1603    0.363936
1604    0.442473
1605    0.425226
          ...   
2129    0.666349
2130    0.613675
2131    0.542096
2132    0.683481
2133    0.816261
Name: y_pred, Length: 533, dtype: float64

array([0.6244654 , 0.64054464, 0.65276626, 0.66692999, 0.63578147,
       0.65462245, 0.67285904, 0.62201025, 0.61897596, 0.62099568])